### 1. 패키지 설치 + 환경변수 로드

In [9]:
%pip install -qU langchain langchain_openai langgraph

Note: you may need to restart the kernel to use updated packages.


In [10]:
from dotenv import load_dotenv
load_dotenv()

True

### 2. checkpointer와 store의 차이

지금까지 쓴 체크포인터는 **thread 안에서만** 유효함. thread를 넘어 유지되는 기억은 `store` 가 담당함

| | `InMemorySaver` (checkpointer) | `InMemoryStore` (store) |
|---|---|---|
| 범위 | thread 단위 (short-term) | thread를 가로지름 (long-term) |
| 저장 단위 | 그래프 State 전체 | namespace + key → value |
| 연결 | `compile(checkpointer=...)` | `compile(store=...)` |
| 키 | `thread_id` | `(user_id, "memories")` 같은 튜플 |

둘은 대체 관계가 아니라 **함께 쓰는** 관계임

### 3. store 단독으로 써보기

In [11]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
namespace = ("user-1", "memories")   # 튜플 형태의 namespace

store.put(namespace, "name", {"memory": "이름은 길동"})
store.put(namespace, "job", {"memory": "직업은 개발자"})

print(store.get(namespace, "name").value)
print([item.value for item in store.search(namespace)])

{'memory': '이름은 길동'}
[{'memory': '이름은 길동'}, {'memory': '직업은 개발자'}]


### 4. 노드에서 store 사용하기

노드가 `Runtime` 을 인자로 받으면 LangGraph가 주입해 줌

In [12]:
from dataclasses import dataclass
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.runtime import Runtime

@dataclass
class Context:
    user_id: str   # 어떤 사용자의 기억을 볼지 결정

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot(state: State, runtime: Runtime[Context]):
    namespace = (runtime.context.user_id, "memories")
    memories = runtime.store.search(namespace)   # thread와 무관하게 조회됨
    info = "\n".join(item.value["memory"] for item in memories)

    system = f"다음은 사용자에 대해 알고 있는 정보입니다:\n{info}"
    answer = llm.invoke([("system", system)] + state["messages"])
    return {"messages": [answer]}

graph_builder = StateGraph(State, context_schema=Context)   # context 타입을 명시
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(checkpointer=InMemorySaver(), store=store)   # 둘 다 연결

### 5. thread "1" 에서 질문

In [13]:
from langchain_core.runnables import RunnableConfig

config_1: RunnableConfig = {"configurable": {"thread_id": "1"}}

result = graph.invoke(
    {"messages": [("user", "내 이름이 뭐야?")]},
    config_1,
    context=Context(user_id="user-1"),
)
print(result["messages"][-1].content)

당신의 이름은 길동입니다.


### 6. thread "2" 에서 같은 질문

03번에서는 thread를 바꾸면 기억하지 못했음. store에 있는 정보는 **thread가 달라도 유지**됨

In [14]:
config_2: RunnableConfig = {"configurable": {"thread_id": "2"}}

result = graph.invoke(
    {"messages": [("user", "내 직업이 뭐라고 했지?")]},
    config_2,
    context=Context(user_id="user-1"),
)
print(result["messages"][-1].content)

당신의 직업은 개발자입니다.


### 7. 다른 사용자

`user_id` 가 다르면 namespace가 달라져 서로의 기억을 보지 못함

In [15]:
result = graph.invoke(
    {"messages": [("user", "내 이름이 뭐야?")]},
    {"configurable": {"thread_id": "3"}},
    context=Context(user_id="user-2"),   # 빈 namespace
)
print(result["messages"][-1].content)

죄송하지만, 당신의 이름은 알 수 없습니다. 사용자에 대한 개인 정보를 보관하지 않기 때문입니다. 이름을 알려주시면 그에 맞춰 대화할 수 있습니다!


### 8. 정리

- `store` 는 thread를 가로지르는 long-term 기억이며 `(namespace, key)` 로 관리함
- 노드는 `Runtime` 을 인자로 받아 `runtime.store` / `runtime.context` 에 접근함
- checkpointer(대화 흐름)와 store(사용자 정보)는 **함께** 사용함
- 운영에서는 `InMemoryStore` 대신 DB 기반 store를 씀
- 참고: [Stores](https://docs.langchain.com/oss/python/langgraph/stores)